In [45]:
# =============================================================================
# Hourly Spatial Interpolation using Ordinary Kriging (Auto-fit)
# =============================================================================
#
# This script constructs hourly 1-km gridded temperature fields using:
#   • Dynamic hourly lapse-rate estimation via linear regression
#   • Ordinary Kriging (OK) on temperature residuals
#   • Per-hour variogram auto-fitting (no fixed parameters)
#
# IMERG is NOT kriged here and will be merged later as a NetCDF.
#
# This approach is intentionally simple and robust for sparse, nonstationary,
# hourly mountain meteorological fields.
#
# =============================================================================

# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------
import xarray as xr
import pandas as pd
import numpy as np
import rasterio as rio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from pyproj import Transformer
from pathlib import Path
from sklearn.linear_model import LinearRegression
from pykrige.ok import OrdinaryKriging
import time
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
BASE_DIR = Path().resolve().parent

CONFIG = {
    "station_hourly": BASE_DIR / "outputs/hourly_pipeline/hourly_data/stations_hourly.parquet",
    "mros_hourly":    BASE_DIR / "outputs/hourly_pipeline/hourly_data/mros_hourly.parquet",
    "dem_path":       BASE_DIR / "DEM_1km_clipped.tif",
    "out_nc":         BASE_DIR / "outputs/hourly_pipeline/kriging_OK_hourly_v6_full.nc",
    "out_dir":  BASE_DIR / "outputs/hourly_pipeline",
    "proj_fallback":  "EPSG:26911"
}

# ---------------------------------------------------------------------
TEMP_VARS = ["temp_air", "temp_dew", "temp_wet"]
OTHER_VARS = ["rh", "mros_plp_proxy"]

PREDICTOR_VARS = TEMP_VARS + OTHER_VARS

MIN_STATIONS_TEMP = 6
MIN_POINTS_RH  = 6
MIN_POINTS_MROS  = 3

GRID_RES_M = 1000  # ~1 km

# # Test Window
# start_time = pd.to_datetime("2025-03-30 00:00")
# end_time   = pd.to_datetime("2025-04-02 23:00")

# Full Window
start_time = pd.to_datetime("2024-10-01 00:00")
end_time   = pd.to_datetime("2025-05-31 23:00")


out_nc = CONFIG["out_nc"]


In [46]:
# ---------------------------------------------------------------------
# Lapse-rate estimation 
# ---------------------------------------------------------------------
def estimate_lapse_rate(
    st_df: pd.DataFrame,
    temp_col: str = "temp_air",
    elev_col: str = "elev",
    default_lapse: float = -0.005,
    min_points: int = 5,
    bounds: tuple = (-0.009, 0.002)
) -> float:
    use = st_df.dropna(subset=[temp_col, elev_col])
    if len(use) < min_points:
        return default_lapse

    X = use[[elev_col]].values.astype(float)
    y = use[temp_col].values.astype(float)

    try:
        model = LinearRegression().fit(X, y)
        slope = model.coef_[0]
        return slope if bounds[0] <= slope <= bounds[1] else default_lapse
    except Exception:
        return default_lapse



In [47]:
# ---------------------------------------------------------------------
# Load DEM and grid
# ---------------------------------------------------------------------
print("\n--- Loading DEM ---")

with rio.open(CONFIG["dem_path"]) as src:
    dem_crs = src.crs
    if not dem_crs or not dem_crs.is_projected:
        print(f"DEM is geographic ({dem_crs}); reprojecting...")
        transform, width, height = calculate_default_transform(
            src.crs, CONFIG["proj_fallback"], src.width, src.height, *src.bounds
        )
        dem_data = np.empty((height, width), dtype=np.float32)
        reproject(
            source=rio.band(src, 1),
            destination=dem_data,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=CONFIG["proj_fallback"],
            resampling=Resampling.bilinear
        )
        dem_profile = src.profile
        dem_profile.update({"crs": CONFIG["proj_fallback"], "transform": transform})
        proj_crs = CONFIG["proj_fallback"]
    else:
        dem_data = src.read(1)
        dem_profile = src.profile
        proj_crs = dem_crs

# Grid centers
T = dem_profile["transform"]
xs = T.c + (np.arange(dem_profile["width"]) + 0.5) * T.a
ys = T.f + (np.arange(dem_profile["height"]) + 0.5) * T.e
gx, gy = np.meshgrid(xs, ys)
grid_xy = np.column_stack([gx.ravel(), gy.ravel()])
grid_elev = dem_data.ravel()

print(f"DEM CRS: {proj_crs}, grid points: {grid_xy.shape[0]:,}")




--- Loading DEM ---
DEM CRS: EPSG:26911, grid points: 35,642


In [48]:
# ---------------------------------------------------------------------
# Load and harmonize observations
print("\n--- Loading observations ---")

st_hr   = pd.read_parquet(CONFIG["station_hourly"])
mros_hr = pd.read_parquet(CONFIG["mros_hourly"])

obs_sets = {
    "stations": st_hr,
    "mros":     mros_hr,
}

# ---------------------------------------------------------------------
# Normalize timestamps (CRITICAL)
# ---------------------------------------------------------------------
print("\n--- Normalizing timestamps (UTC → tz-naive hourly) ---")

for name, df in obs_sets.items():
    if "hour_utc" not in df.columns:
        raise RuntimeError(f"{name} missing required column: hour_utc")

    df["hour_utc"] = (
        pd.to_datetime(df["hour_utc"], utc=True, errors="coerce")
        .dt.floor("h")
        .dt.tz_localize(None)
    )

    n_bad = df["hour_utc"].isna().sum()
    if n_bad > 0:
        raise RuntimeError(f"{name}: {n_bad} rows have invalid timestamps")

    print(
        f"{name:8s} | "
        f"time: {df['hour_utc'].min()} → {df['hour_utc'].max()} | "
        f"rows: {len(df):,}"
    )

# ---------------------------------------------------------------------
# Reproject coordinates to DEM / grid CRS
# ---------------------------------------------------------------------
print("\n--- Reprojecting observation coordinates ---")

transformer = Transformer.from_crs("EPSG:4326", proj_crs, always_xy=True)

def project_xy(df, name):
    if not {"lon", "lat"}.issubset(df.columns):
        raise RuntimeError(f"{name} missing lon/lat columns")

    x, y = transformer.transform(df["lon"].values, df["lat"].values)
    df["x"] = x
    df["y"] = y

    if not np.isfinite(x).all() or not np.isfinite(y).all():
        raise RuntimeError(f"{name}: non-finite projected coordinates detected")

    print(
        f"{name:8s} | "
        f"x[{x.min():.0f}, {x.max():.0f}]  "
        f"y[{y.min():.0f}, {y.max():.0f}]"
    )

project_xy(st_hr,   "stations")
project_xy(mros_hr, "mros")

# Final structural checks (fail fast)
print("\n--- Final observation checks ---")

required_station_cols = {"hour_utc", "x", "y", "elev"}
missing = required_station_cols - set(st_hr.columns)
if missing:
    raise RuntimeError(f"stations missing required columns: {missing}")

required_mros_cols = {"hour_utc", "x", "y", "mros_plp_proxy"}
missing = required_mros_cols - set(mros_hr.columns)
if missing:
    raise RuntimeError(f"mros missing required columns: {missing}")

print("✔ Observation tables validated and ready for kriging")



--- Loading observations ---

--- Normalizing timestamps (UTC → tz-naive hourly) ---
stations | time: 2024-10-01 00:00:00 → 2025-05-31 23:00:00 | rows: 557,766
mros     | time: 2024-10-06 21:00:00 → 2025-05-18 02:00:00 | rows: 7,896

--- Reprojecting observation coordinates ---
stations | x[144827, 410147]  y[3930064, 4432428]
mros     | x[147857, 407691]  y[3935481, 4429322]

--- Final observation checks ---
✔ Observation tables validated and ready for kriging


In [49]:
# ---------------------------------------------------------------------
# Hour list
# ---------------------------------------------------------------------
hours = pd.date_range(start=start_time, end=end_time, freq="h")

print("\n================================================================================")
print("Building hourly processing list")
print("================================================================================")
print(f"Hours to process: {len(hours)}")
print(f"First hour: {hours[0]} | Last hour: {hours[-1]}")

# ---------------------------------------------------------------------
# Output dataset (INITIALIZE ONCE)
# ---------------------------------------------------------------------
PREDICTOR_VARS = [
    "temp_air",
    "temp_dew",
    "temp_wet",
    "rh",
    "mros_plp_proxy",
]

out_ds = xr.Dataset(
    coords=dict(
        time=hours,
        y=ys,
        x=xs,
    )
)

for v in PREDICTOR_VARS:
    out_ds[v] = (
        ("time", "y", "x"),
        np.full((len(hours), len(ys), len(xs)), np.nan, dtype=np.float32),
    )


# Functions -----------------------------------------------------------

def run_ok_kriging(
    x, y, values,
    grid_xy,
    variogram_model="spherical"
):
    OK = OrdinaryKriging(
        x, y, values,
        variogram_model=variogram_model,
        enable_plotting=False,
        verbose=False
    )

    z, _ = OK.execute("points", grid_xy[:, 0], grid_xy[:, 1])
    return z, OK.variogram_model_parameters

def run_hour(
    hr,
    ti,
    st_hr,
    mros_hr,
    out_ds,
    grid_xy,
    grid_elev
):
    print(f"\n--- Hour {hr} ---")

    # -----------------------------
    # Station data for this hour
    stn = st_hr[st_hr["hour_utc"] == hr]
    print(f"  n(stations): {len(stn)}")

    # -----------------------------
    # Temperature variables
    if len(stn) >= MIN_STATIONS_TEMP:
        lapse = estimate_lapse_rate(stn)
        print(f"  lapse rate: {lapse*1000:.2f} °C/km")

        for v in TEMP_VARS:
            sub = stn.dropna(subset=[v, "elev", "x", "y"])
            if len(sub) < MIN_STATIONS_TEMP:
                print(f"   {v}: insufficient points ({len(sub)})")
                continue

            resid = sub[v].values - lapse * sub["elev"].values

            try:
                z_resid, vg = run_ok_kriging(
                    sub["x"].values,
                    sub["y"].values,
                    resid,
                    grid_xy
                )
                z_full = z_resid + lapse * grid_elev
                out_ds[v][ti, :, :] = z_full.reshape(len(ys), len(xs))

                print(f"   {v}: OK success | variogram={vg}")

            except Exception as e:
                print(f"   {v}: kriging failed → {e}")

    else:
        print("  Not enough stations for temperature variables")

    # -----------------------------
    # RH (station-based)
    rh_sub = stn.dropna(subset=["rh", "x", "y"])
    if len(rh_sub) >= MIN_POINTS_RH:
        try:
            z, vg = run_ok_kriging(
                rh_sub["x"].values,
                rh_sub["y"].values,
                rh_sub["rh"].values,
                grid_xy
            )
            out_ds["rh"][ti, :, :] = z.reshape(len(ys), len(xs))
            print(f"   rh: OK success | variogram={vg}")
        except Exception as e:
            print(f"   rh: kriging failed → {e}")
    else:
        print(f"   rh: insufficient points ({len(rh_sub)})")

    # -----------------------------
    # MRoS proxy
    mros = mros_hr[mros_hr["hour_utc"] == hr]
    print(f"  n(MRoS): {len(mros)}")

    mros_sub = mros.dropna(subset=["mros_plp_proxy", "x", "y"])
    if len(mros_sub) >= MIN_POINTS_MROS:
        try:
            z, vg = run_ok_kriging(
                mros_sub["x"].values,
                mros_sub["y"].values,
                mros_sub["mros_plp_proxy"].values,
                grid_xy
            )
            out_ds["mros_plp_proxy"][ti, :, :] = z.reshape(len(ys), len(xs))
            print(f"   mros_plp_proxy: OK success | variogram={vg}")
        except Exception as e:
            print(f"   mros_plp_proxy: kriging failed → {e}")
    else:
        print(f"   mros_plp_proxy: insufficient points ({len(mros_sub)})")



Building hourly processing list
Hours to process: 5832
First hour: 2024-10-01 00:00:00 | Last hour: 2025-05-31 23:00:00


In [50]:
print("\n================================================================================")
print("Starting hourly OK auto-fit kriging")
print("================================================================================")

for ti, hr in enumerate(hours):
    t0 = time.time()

    run_hour(
        hr=hr,
        ti=ti,
        st_hr=st_hr,
        mros_hr=mros_hr,
        out_ds=out_ds,
        grid_xy=grid_xy,
        grid_elev=grid_elev
    )

    print(f"  Completed hour in {time.time() - t0:.1f} s")


print("\n=== Population summary ===")
for v in PREDICTOR_VARS:
    n_ok = np.isfinite(out_ds[v].values).any(axis=(1,2)).sum()
    print(f"{v:16s}: {n_ok}/{len(hours)} hours populated")


Starting hourly OK auto-fit kriging

--- Hour 2024-10-01 00:00:00 ---
  n(stations): 311
  lapse rate: -6.31 °C/km
   temp_air: OK success | variogram=[3.45442236e+01 4.85430865e+05 2.01516290e+00]
   temp_dew: OK success | variogram=[4.79763794e+00 3.92946083e+05 2.42735059e+00]
   temp_wet: OK success | variogram=[2.27721015e+01 4.85430865e+05 4.23269567e+00]
   rh: OK success | variogram=[1.06244489e+01 1.64781232e+04 3.59982957e+01]
  n(MRoS): 0
   mros_plp_proxy: insufficient points (0)
  Completed hour in 4.4 s

--- Hour 2024-10-01 01:00:00 ---
  n(stations): 310
  lapse rate: -6.00 °C/km
   temp_air: OK success | variogram=[3.30214468e+01 4.85430865e+05 2.44518851e+00]
   temp_dew: OK success | variogram=[6.05569125e+00 2.64931891e+05 3.03975234e+00]
   temp_wet: OK success | variogram=[3.55610271e+01 4.85430865e+05 4.90542055e-01]
   rh: OK success | variogram=[3.13680016e-03 3.77790147e+04 6.48267168e+01]
  n(MRoS): 0
   mros_plp_proxy: insufficient points (0)
  Completed hou

In [51]:
# -------------------- Save NetCDF ------------------------------------

# Ensure time is tz-naive (xarray-safe)
ds = out_ds.assign_coords(time=pd.to_datetime(out_ds.time.values).astype("datetime64[ns]"))

# Attach spatial metadata from DEM
ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
ds = ds.rio.write_crs(dem_profile["crs"])
ds = ds.rio.write_transform(dem_profile["transform"])

# Link grid mapping
for v in ds.data_vars:
    ds[v].attrs.setdefault("grid_mapping", "spatial_ref")

# Compression / chunking (simple + safe)
encoding = {
    v: {"zlib": True, "complevel": 4}
    for v in ds.data_vars
}

ds.to_netcdf(out_nc, engine="netcdf4", encoding=encoding)
print(f"Wrote NetCDF: {out_nc}")
print("CRS:", ds.rio.crs)
print("Dims:", dict(ds.sizes))


Wrote NetCDF: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\kriging_OK_hourly_v6_full.nc
CRS: EPSG:26911
Dims: {'y': 251, 'x': 142, 'time': 5832}


In [52]:
# -------------------- Quick Plotting ------------------------------------

from pyproj import CRS


def print_time(ts):
    return pd.to_datetime(ts).strftime("%Y-%m-%d %H:%MZ")

def quicklook_hour(
    ds, t, st_t, mros_t, out_png,
    vars_to_show=("mros_plp_proxy","temp_air","temp_dew","temp_wet","rh")
):
    # match time index
    times_ds = pd.to_datetime(ds.time.values).floor("h")
    t_floor  = pd.to_datetime(t).floor("h")
    if t_floor not in times_ds.values:
        print(f"No matching time {t_floor} in dataset for quicklook.")
        return
    ti = int(np.where(times_ds == t_floor)[0][0])

    # axes extent (xmin, xmax, ymin, ymax)
    xvals = ds["x"].values
    yvals = ds["y"].values
    xmin, xmax = float(np.min(xvals)), float(np.max(xvals))
    ymin, ymax = float(np.min(yvals)), float(np.max(yvals))
    extent = [xmin, xmax, ymin, ymax]

    keep = [v for v in vars_to_show if v in ds.data_vars]
    if not keep:
        print("No matching variables to plot.")
        return
    ncols, nrows = 3, int(np.ceil(len(keep)/3))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 3.8*nrows), squeeze=False)
    fig.suptitle(f"Quicklook @ {t_floor:%Y-%m-%d %H:%MZ}", fontsize=14)

    # dataset CRS (fallback to configured)
    target_crs = ds.rio.crs or CRS.from_user_input(CONFIG["proj_fallback"])
    tf = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)

    # --- project & CLIP stations ---
    st_x = np.empty(0)
    st_y = np.empty(0)
    if len(st_t):
        sx, sy = tf.transform(st_t["lon"].values, st_t["lat"].values)
        sx = np.asarray(sx); sy = np.asarray(sy)
        smask = (sx >= xmin) & (sx <= xmax) & (sy >= ymin) & (sy <= ymax) & np.isfinite(sx) & np.isfinite(sy)
        st_x, st_y = sx[smask], sy[smask]

    # --- project & CLIP MRoS ---
    mo_x = np.empty(0)
    mo_y = np.empty(0)
    if len(mros_t):
        mx, my = tf.transform(mros_t["lon"].values, mros_t["lat"].values)
        mx = np.asarray(mx); my = np.asarray(my)
        mmask = (mx >= xmin) & (mx <= xmax) & (my >= ymin) & (my <= ymax) & np.isfinite(mx) & np.isfinite(my)
        mo_x, mo_y = mx[mmask], my[mmask]

    for i, var in enumerate(keep):
        ax = axes[i // ncols, i % ncols]
        arr = ds[var].isel(time=ti).values

        # color scaling
        if var in ("plp", "mros_plp_proxy", "rh"):
            im = ax.imshow(arr, origin="upper", extent=extent, aspect="equal", vmin=0, vmax=100)
        else:
            im = ax.imshow(arr, origin="upper", extent=extent, aspect="equal")

        ax.set_title(var)
        ax.set_xlabel("Easting (km)")
        ax.set_ylabel("Northing (km)")
        ax.ticklabel_format(style="plain")   # disable 1e6 scientific format
        ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
        ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])

        # overlay
        if st_x.size:
            ax.scatter(st_x, st_y, s=15, c="white", edgecolor="k",
                    marker="o", linewidths=0.5, label="Stations", zorder=3)
        if mo_x.size:
            ax.scatter(mo_x, mo_y, s=25, c="red", edgecolor="k",
                    marker="^", linewidths=0.6, label="MRoS", zorder=3)

        ax.legend(loc="upper right", frameon=True, fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

    # turn off any leftover panels
    for j in range(len(keep), nrows*ncols):
        axes[j // ncols, j % ncols].axis("off")

    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    fig.savefig(out_png, dpi=200)
    plt.close(fig)
    print(
        f"Saved quicklook: {out_png} | plotted {st_x.size} stations, {mo_x.size} MRoS (clipped to DEM)"
    )


# -------------------- Loop --------------------

quick_dir = Path(CONFIG["out_dir"]) / "maps"
quick_dir.mkdir(parents=True, exist_ok=True)

import rioxarray
from pyproj import CRS

# Ensure CRS exists for plotting
if not hasattr(out_ds, "rio") or out_ds.rio.crs is None:
    out_ds = out_ds.rio.write_crs(CONFIG["proj_fallback"])

# Normalize obs time ONCE
st_hr["hour_utc"]   = pd.to_datetime(st_hr["hour_utc"]).dt.floor("h")
mros_hr["hour_utc"] = pd.to_datetime(mros_hr["hour_utc"]).dt.floor("h")

sample_hours = pd.to_datetime(out_ds.time.values)[::max(1, len(out_ds.time)//20)]

for t in sample_hours:
    t_floor = pd.to_datetime(t).floor("h")

    st_t   = st_hr[st_hr["hour_utc"] == t_floor]
    mros_t = mros_hr[mros_hr["hour_utc"] == t_floor]

    print(f"[{t_floor}] Stations: {len(st_t)}, MRoS: {len(mros_t)}")

    quicklook_hour(
        out_ds,
        t_floor,
        st_t,
        mros_t,
        out_png=quick_dir / f"kriging_v6_full_{print_time(t_floor).replace(':','-')}.png"
    )


[2024-10-01 00:00:00] Stations: 311, MRoS: 0


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_9572\3998957926.py:73: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_9572\3998957926.py:74: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_9572\3998957926.py:73: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_9572\3998957926.py:74: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\kriging_v6_full_2024-10-01 00-00Z.png | plotted 127 stations, 0 MRoS (clipped to DEM)
[2024-10-13 03:00:00] Stations: 349, MRoS: 0


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_9572\3998957926.py:73: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_9572\3998957926.py:74: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_9572\3998957926.py:73: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_9572\3998957926.py:74: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\kriging_v6_full_2024-10-13 03-00Z.png | plotted 157 stations, 0 MRoS (clipped to DEM)
[2024-10-25 06:00:00] Stations: 348, MRoS: 0


...

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\kriging_v6_full_2025-05-07 06-00Z.png | plotted 40 stations, 0 MRoS (clipped to DEM)
[2025-05-19 09:00:00] Stations: 55, MRoS: 0


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_9572\3998957926.py:73: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_9572\3998957926.py:74: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_9572\3998957926.py:73: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_9572\3998957926.py:74: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\kriging_v6_full_2025-05-19 09-00Z.png | plotted 38 stations, 0 MRoS (clipped to DEM)
[2025-05-31 12:00:00] Stations: 340, MRoS: 0


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_9572\3998957926.py:73: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_9572\3998957926.py:74: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_9572\3998957926.py:73: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_9572\3998957926.py:74: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\kriging_v6_full_2025-05-31 12-00Z.png | plotted 158 stations, 0 MRoS (clipped to DEM)
